In [1]:
# All Necessary Imports
import numpy as np
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import re
from textblob import TextBlob
from wordcloud import WordCloud
import seaborn as sns
import matplotlib.pyplot as plt
import cufflinks as cf
%matplotlib inline
from plotly.offline import init_notebook_mode, iplot
init_notebook_mode(connected = True)
cf.go_offline();
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns",None)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Download NLTK data (run this only once)
nltk.download('stopwords')
nltk.download('wordnet')

# Load and Prepare the Data
df = pd.read_csv("amazon.csv")
df = df.sort_values("wilson_lower_bound", ascending=False)
df.drop("Unnamed: 0", inplace=True, axis=1)

# Advanced Data Cleaning and Column Creation
def advanced_clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower()
    words = text.split()
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    cleaned_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return " ".join(cleaned_words)

df['reviewText_cleaned'] = df['reviewText'].apply(advanced_clean_text)
df['sentiment_class'] = df['overall'].apply(lambda x: 'Positive' if x >= 4 else 'Negative' if x <= 2 else 'Neutral')

print("Sentiment Class Distribution:")
print(df['sentiment_class'].value_counts())

# Vectorization and Model Training
X = df['reviewText_cleaned']
y = df['sentiment_class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
logistic_model = LogisticRegression(solver='liblinear')
logistic_model.fit(X_train_vec, y_train)

print("\nLogistic Regression model trained successfully!")

# Model Evaluation
y_pred = logistic_model.predict(X_test_vec)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sathv\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sathv\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Sentiment Class Distribution:
sentiment_class
Positive    4449
Negative     324
Neutral      142
Name: count, dtype: int64

Logistic Regression model trained successfully!

Classification Report:
              precision    recall  f1-score   support

    Negative       1.00      0.25      0.40        65
     Neutral       0.00      0.00      0.00        28
    Positive       0.92      1.00      0.96       890

    accuracy                           0.92       983
   macro avg       0.64      0.42      0.45       983
weighted avg       0.90      0.92      0.89       983

Accuracy: 0.92
